# 0.5 Lab: structured-output classifier

AI Agent Engineering · LogicStack — practice notebook

Run the cells in order. Change the code, break things, fix them. That is the practice.

Lesson page: https://logicstack.org/courses/ai-agent-engineering

In [ ]:
# Run this first: installs the exact versions this course uses.
!pip install -q "langchain==1.4.7" "langchain-core==1.6.13" "langchain-openai==1.7.2" "langchain-google-genai==4.4.2" "langgraph==1.2.16"

## Your API key - pick ONE option

Some cells in this notebook call a model, and you need your own API key. Run only ONE of the next two cells, whichever fits you:

**Option A - Gemini (free, recommended if you have no OpenAI credit):** Google AI Studio gives a free API key, no card needed. Open https://aistudio.google.com/apikey , create a key, then run the Gemini cell below. The Flash models run on a free tier with rate limits.

**Option B - OpenAI (paid):** create a key at https://platform.openai.com/api-keys and run the OpenAI cell below.

Whichever key you paste is stored only in this notebook session, never saved to the file.

In [ ]:
# Option A: Gemini (free). Run this cell OR the OpenAI one, not both.
import os, getpass
os.environ["GOOGLE_API_KEY"] = getpass.getpass("Paste your Gemini (AI Studio) API key (input stays hidden): ")

In [ ]:
# Option B: OpenAI (paid). Run this cell OR the Gemini one, not both.
import os, getpass
os.environ["OPENAI_API_KEY"] = getpass.getpass("Paste your OpenAI API key (input stays hidden): ")

## Using Gemini instead of OpenAI in the cells below

If you ran the Gemini key cell, one more small step: wherever a cell below creates the model with `ChatOpenAI(...)`, swap that creation line for:

```python
from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")
```

(Use whatever variable name the cell used, for example `model = ...` instead of `llm = ...`.) Everything else stays the same: prompts, chains, parsers and tools do not care which provider the model came from. For embeddings, swap `OpenAIEmbeddings` for `GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")` from the same package. On the OpenAI path, ignore this note and run the cells exactly as written.

## Learning objectives

- Build a text classifier that returns a validated Pydantic object via `with_structured_output`.
- Write Pydantic schemas that models actually obey: descriptions on every field, `Literal` enums for labels, an escape hatch for ambiguity.
- Inspect and handle parse failures with `include_raw=True`.
- Run the classifier over a batch of inputs with `.batch()` and score it against hand-labelled data.

## The idea in plain words

In lesson 0.4 you learned the three prompting layers: instruct, demonstrate, enforce. This lab is the enforce layer made concrete. You will build a support-ticket classifier with zero training data: a Pydantic model plus one good schema description does the work that used to need a labelled dataset and a scikit-learn pipeline.

Before we start, be honest about the trade-off, because a researcher-audience course should not pretend LLMs are free. On a hosted model like gpt-4o-mini, classifying a ticket costs a fraction of a cent; at a million tickets that is real money (tens of dollars), while a trained sklearn classifier costs near-zero CPU per prediction. The LLM classifier wins when you have no labelled data, when inputs are multilingual or adversarially phrased, and when labels change often. The sklearn classifier wins on pure unit cost at fixed volume. Know which game you are playing.

## Step 1: the schema

Create `triage.py` in your course folder. The schema is the specification; get it right and the rest is plumbing.

In [ ]:
from typing import Literal
from pydantic import BaseModel, Field

class TicketTriage(BaseModel):
    """Classify a customer support ticket so it can be routed to the right team."""

    category: Literal["billing", "technical", "account", "other"] = Field(
        description=(
            "billing: anything about charges, invoices, refunds, payments. "
            "technical: bugs, crashes, error messages, things that do not work. "
            "account: login, password, profile, subscription plan changes. "
            "other: only when none of the above clearly fits."
        )
    )
    confidence: float = Field(
        description="Your confidence in this classification, between 0 and 1."
    )
    reason: str = Field(
        description="One sentence explaining which words in the ticket drove your decision."
    )
    escalate: bool = Field(
        description="True if the ticket sounds urgent, angry, or like a legal threat."
    )

Notice the `category` description does the classification work: each label gets a plain-English definition with examples. That description is prompt text, so write it like documentation, not like a comment.

One judgement call: `confidence` is a bare `float`, not `Field(ge=0.0, le=1.0)`. Under OpenAI's `strict=True` mode, numeric metadata like `ge`/`le` is rejected by the API. The description carries the bounds instead. If you need hard numeric validation, add it in Python after parsing.

## Step 2: the classifier

Add the chain. Model construction is one line, and swapping providers later is a two-line change (see lesson 0.2 for the Ollama path):

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

structured_llm = llm.with_structured_output(
    TicketTriage,
    method="json_schema",   # native structured output: guaranteed schema-conformant
    strict=True,
)

prompt = ChatPromptTemplate.from_messages([
    ("system", "Classify support tickets. Be precise and conservative. Use 'other' when unsure."),
    ("human", "{ticket}"),
])

classifier = prompt | structured_llm

def classify(ticket: str) -> TicketTriage:
    ticket = ticket.strip()
    if not ticket:
        raise ValueError("Ticket text must not be empty.")
    if len(ticket) > 2000:
        raise ValueError("Ticket text is too long; keep it under 2000 characters.")
    return classifier.invoke({"ticket": ticket})

if __name__ == "__main__":
    result = classify("I was charged twice for my subscription this month")
    print(result)
    # TicketTriage(category='billing', confidence=0.97, reason='The ticket reports a duplicate charge.', escalate=False)

`temperature=0` matters here: you want the most likely label every time, not a creative one. And the input validation at the top of `classify` is deliberate. Lesson 0.4's rule was "validate tool-derived values in Python before using them"; the same applies to user input. An empty ticket sent to a paid API is a wasted call with a garbage answer.

## Step 3: see failures, not just successes

Happy-path code hides the failure modes. Flip on `include_raw` in a scratch script and watch what happens on adversarial input:

In [ ]:
debug_classifier = (prompt | llm.with_structured_output(
    TicketTriage, method="json_schema", strict=True, include_raw=True
))

for ticket in [
    "I was charged twice for my subscription",
    "",                                    # caught by our validation before the API
    "asdf jkl qwerty",                     # nonsense in
]:
    try:
        out = debug_classifier.invoke({"ticket": ticket})
    except ValueError as e:
        print(f"rejected locally: {e}")
        continue
    if out["parsing_error"]:
        print(f"PARSE FAILURE on {ticket!r}")
        print(out["raw"].content)           # what the model actually emitted
    else:
        print(f"OK: {out['parsed'].category} ({out['parsed'].confidence:.2f})")

Two habits to take from this. First, `include_raw=True` turns "the chain crashed" into "here is exactly what the model emitted and why parsing failed", which is the difference between debugging in minutes and debugging in days. Second, notice the failure layers are ordered: your Python validation catches bad input before money is spent; schema enforcement catches bad labels at the provider; `include_raw` catches whatever slips through both.

When a parse error does reach you, the sanctioned recovery is a small bounded retry: re-invoke once or twice with a "reply with JSON only" instruction appended, then give up and escalate. `with_structured_output` has no built-in retry parameter, so write the loop yourself and cap it. Transport errors (timeouts, 429s) are different: those retry with the model's `max_retries` constructor argument or `.with_retry()` on the runnable.

## Step 4: batch and score

The classifier is a standard Runnable, so batching needs no extra code. `.batch()` runs inputs concurrently:

In [ ]:
tickets = [
    "My invoice shows the wrong VAT amount",
    "The app freezes when I open settings",
    "I cannot reset my password, the email never arrives",
    "You charged my old card after I updated it",
    "How do I export my data to CSV?",
]

results = classifier.batch([{"ticket": t} for t in tickets])
for ticket, r in zip(tickets, results):
    print(f"{r.category:10s} {r.confidence:.2f}  {ticket[:50]}")

Now the part most tutorials skip: a ten-line self-check that tells you whether the thing actually works. Hand-label ten tickets, including two tricky ones, and count matches:

In [ ]:
TEST_SET = [
    ("I was charged twice", "billing"),
    ("Cancel my invoice and refund me", "billing"),      # tricky: mentions account-ish words
    ("The dashboard shows a blank page", "technical"),
    ("Password reset link is broken", "account"),        # tricky: broken link could read as technical
    ("Where is your office located?", "other"),
    ("My card was declined at checkout", "billing"),
    ("The API returns 500 on POST /orders", "technical"),
    ("Change my plan from monthly to yearly", "account"),
    ("I will sue you if you do not fix this", "other"), # escalate=True expected too
    ("Thanks for the quick fix yesterday", "other"),
]

correct = 0
for text, expected in TEST_SET:
    got = classify(text).category
    ok = got == expected
    correct += ok
    if not ok:
        print(f"MISMATCH: expected {expected!r}, got {got!r} for {text!r}")

print(f"accuracy: {correct}/{len(TEST_SET)}")

Run it, look at the mismatches, and decide: is the label wrong, or is the schema description ambiguous? Most "model errors" at this stage are schema errors. Fix the description, re-run, watch the score move. That loop, description to score to description, is the core skill of this lab, and it is exactly how you will tune retrieval prompts in Module 3.

## Stretch: cost a real workload

Take the batch from Step 4 and add token usage to the loop. LangChain surfaces it on the raw message:

In [ ]:
out = debug_classifier.invoke({"ticket": "I was charged twice for my subscription"})
usage = out["raw"].usage_metadata
print(usage)  # {'input_tokens': ..., 'output_tokens': ..., 'total_tokens': ...}

Price one ticket from the provider's public pricing page, multiply by your expected volume, and compare against the sklearn alternative honestly. If the LLM is ten times more expensive but saves you three weeks of labelling, say that out loud. Engineering decisions with numbers beat vibes.

## What can go wrong

- **The model invents labels.** Fixed by `Literal` + strict structured output, as built here. If you see a label outside the set, your `method` is not enforcing, check it first.
- **Confidence is miscalibrated.** Model-reported confidence is not a probability; it correlates with certainty only loosely. Never use it as a threshold for irreversible actions without measuring calibration on your own data first.
- **Weak local models break the schema.** Small Ollama models may narrate tool calls as prose instead of emitting valid JSON. Shrink the schema to the minimum fields, prefer `json_mode`/`json_schema` over function-calling for local models, and test the exact model you will ship with, not the hosted one you developed against.
- **Ambiguous tickets get forced into wrong labels.** That is what `"other"` and `reason` are for. Read the reasons in your test-set mismatches; they tell you where your label definitions are fuzzy.
- **You tested ten tickets and shipped.** The ten-ticket set catches schema bugs, not distribution shift. Real ticket streams contain languages, typos, and anger your test set never imagined. Grow the set from production traffic (Module 6 covers golden datasets properly).

## Check your understanding

**1. Why is `temperature=0` set on the classifier's model?**
Classification needs the most likely label deterministically, not a creative sample. Temperature 0 removes sampling randomness so the same ticket gives the same label across runs, which is also what makes the test set reproducible.

**2. `include_raw=True` changes the chain's output. What does it return, and when would you use it?**
It returns a dict with `raw` (the AIMessage), `parsed` (the validated object), and `parsing_error`. Use it during development and in error logging: it shows exactly what the model emitted when parsing fails, instead of raising an opaque exception.

**3. A ticket reading "Cancel my invoice" gets classified as "account" instead of "billing". Where do you fix this, and why there first?**
In the schema's `Field(description=...)` for `category`: clarify that invoice cancellation and refunds are billing, even when the wording mentions accounts. Most classification "model errors" at this stage are ambiguous label definitions, and the description is the cheapest place to fix them. Re-run the test set to confirm.

**4. Name the two failure layers that run before the model is even called in `classify()`.**
Python input validation (non-empty, length cap) and the decision to send anything at all. They catch bad input before a paid API call is made, so failures are cheap and local.

**5. Your workload is 2 million tickets a month. When would you replace this LLM classifier with a trained sklearn model?**
When labels are stable, you have (or can cheaply build) a few hundred labelled examples per class, inputs are in one language and mostly well-formed, and the per-call API cost at that volume outweighs the labelling and retraining effort. Keep the LLM classifier as the fallback for low-confidence or "other" cases.

## Done

Back to the lesson: https://logicstack.org/courses/ai-agent-engineering

When the code in this notebook feels easy, mark the lesson complete and move to the next one.